In [ ]:
# -----------------------------导入python库------------------------------------
import os
import netCDF4 as nc
import pandas as pd
import numpy as np
import numpy.ma as ma
import xarray as xr
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.style as mplstyle
import cartopy.crs as ccrs
import netCDF4 as nc
from netCDF4 import Dataset
import shapefile
import time
from datetime import datetime, timedelta

import cmaps
BASE_PATH = "/home/yfdong/data/work/SMP"
DATA_PATH = "/raid61/yfdong/data/work/SMP"
import sys
# 将目标模块所在的路径添加到 sys.path 中
sys.path.append(os.path.join(BASE_PATH, "code", "Library"))
from MeteoVarPlot import draw_2DVAR, draw_Bias, draw_VARtemporal, draw_Bias_subplot,draw_2DVAR_subplot ,draw_2DuvVAR ,draw_uvBias_subplot ,draw_2DuvVAR_subplot
from MeteoChartPlot import add_TickGrid
from PreprocessVar import (
    get_MONTH_abbr, get_pentad, get_letter,
    preprocess_var, read_var,Events_configs,
    Read_VarsFromVarsDict, experiments_types,return_Event_Date,
    load_geodata, SubsetDomainConfigure ,create_domain_mask,get_letter,
    
)
Era5Land_VarResults={}
Wrfout_VarResults={}
Obs_DataPath = os.path.join(DATA_PATH, "Evaluation/Obs")
Sim_DataPath = os.path.join(DATA_PATH, "Evaluation/WRF")
# Era5Land_VarList = ['sp','u10','v10','t2m','q2m','tp','e','swvl1']
# Wrfout_VarList = ['PSFC', 'U10', 'V10', 'T2', 'Q2', 'RAINNC', 'ET', 'SH2O_Layer1']
# -----------------------------读取数据------------------------------------
# LSM_VarList = ['T2m', 'Q2m', 'Prec', 'SM', 'ET', 'SH', 'PSFC'] #'Wind10m', 'T2m', 'PSFC', 'ET', 'Q2m', 'Wind10m', 'Prec'
LSM_VarList = ['T2max', 'T2min', 'Prec'] #'Wind10m', 'T2m', 'PSFC', 'ET', 'Q2m', 'Wind10m', 'Prec', 'SPEI'
# LSM_VarList = ['Prec']
# 读取地形数据
# LAT, LON, LandMask, DEM = load_geodata(GeoPath =os.path.join(DATA_PATH, "Domain/geo_em.d01.nc") ,IF_LandMask=True)
ds = xr.open_dataset('/raid61/yfdong/data/work/SMP/Evaluation/WRF/Ctrl_ET_10km.2013.nc')
lat, lon = ds['lat'], ds['lon']
lon_grid, lat_grid = np.meshgrid(lon, lat)
et_2d = ds['ET'][0, :, :].values
LandMask = xr.where(et_2d != 0, 1.0, np.nan)
Timedelta = 24  # 选取的wrfout时间间隔（hour）
CalibrationStatus = 'def'
LAYER = 0
START_YEAR = 2000
END_YEAR = 2024
# YearNum = END_YEAR - START_YEAR + 1
# Years = [2016, 2020]
Years = list(range(START_YEAR, END_YEAR + 1))
YearNum = len(Years)
StartReadMonth, EndReadMonth = 6, 8
StartDay, EndDay = 1, 31
CtrlName, ExpName = 'Ctrl', 'FSM'

In [ ]:
obs_var_mapping = {
    'ET': 'E',
    'SM': 'SMs',  # 土壤湿度第一层
    'SH': 'H',
    'Q2m': 'shum',
    'T2m': 'temp',
    'T2max': 'temp_max',
    'T2min': 'temp_min',
    'PSFC': 'pres',
    'Wind10m': 'wind',
    'Prec': 'cmorph',
    'SPEI': 'SPEI_1d',
}
sim_var_mapping = {
    'SM': 'SH2O_Layer1',  # 土壤湿度第一层
    'ET': 'ET',
    'SH': 'SH',
    'Q2m': 'Q2',
    'T2m': 'T2',
    'T2max': 'T2max',
    'T2min': 'T2min',
    'PSFC': 'PSFC',
    'Wind10m': 'Wind10',
    'Prec': 'RAINNC',
    'SPEI': 'SPEI_1d',
}
obs_folder_mapping = {
    'ET': 'GLEAM',
    'SH': 'GLEAM', #GLEAM_H_wrfout_grid.2021
    'SM': 'GLEAM',
    'Q2m': 'CMFD', 
    'T2m': 'CMFD',
    'T2max': 'CMFD',
    'T2min': 'CMFD',
    'PSFC': 'CMFD', #CMFD_pres_wrfout_grid.2018.nc
    'Wind10m': 'CMFD',
    'Prec': 'CMORPH', 
    'SPEI': 'GLEAM', 
}
sim_folder_mapping = {
    'ET': 'ET',
    'LH': 'LH',
    'SH': 'SH',
    'SM': 'SH2O_Layer1',  # 土壤湿度第一层
    'RSM': 'SH2O_Layer3',  # 土壤湿度第三层
    # 'SST': 'TSLB_Layer1',  # 土壤温度第一层
    'Q2m': 'Q2',
    'T2m': 'T2',
    'T2max': 'T2max',
    'T2min': 'T2min',
    'PSFC': 'PSFC',
    'Wind10m': 'Wind10',
    'Prec':'RAINNC',
    'SPEI': 'SPEI_1d',
}

obs_var_ScaleFactor = {
    'ET': 1,
    'SH': 1,
    'SM': 1,
    'Q2m': 1000, # kg/kg -> g/kg
    'Wind10m': 1,
    'T2m': 1,
    'T2max': 1,
    'T2min': 1,
    'Prec': 1, #
    'PSFC': 0.01, # Pa -> hPa
    'SPEI': 1,
}
sim_var_ScaleFactor = {
    'ET': 1,
    'SH': 1,
    'Q2m': 1, # 
    'SM': 1,
    'Wind10m': 1,
    'T2m': 1,
    'T2max': 1,
    'T2min': 1,
    'Prec': 24, # mm/h -> mm/d
    'PSFC': 1, 
    'SPEI': 1,
}

VarUnit = {
    'SM': 'm³/m³',
    'ET': 'mm/day',
    'SH': 'W/m²',
    'Q2m': 'g/kg',
    'T2m': 'K',
    'T2max': 'K',
    'T2min': 'K',
    'PSFC': 'hPa',
    'Wind10m': 'm/s',
    'Prec': 'mm/day',
    'SPEI': '-',
}
# 子区域划分
SubsetRegions = {
    'HHP':[35, 40, 114.5, 119],
    'LPT': [35, 40, 108, 114.5],
    'QDW': [28.5, 35, 108, 112.5], # Qinling-Daba-Wushan
    'YHP': [28.5, 35, 112.5, 122], 
    'SEH': [23, 28.5, 110, 120] # Southeastern Hills
}
def compute_metric(obs, sim, metric_name):
    # 质控（也可在外部做）
    obs = np.asarray(obs, dtype=np.float64)
    sim = np.asarray(sim, dtype=np.float64)
    valid = ~(np.isnan(obs) | np.isnan(sim))
    if not np.any(valid):
        return np.nan
    obs, sim = obs[valid], sim[valid]
    if metric_name == 'RMSE':
        return np.sqrt(np.nanmean((sim - obs) ** 2))
    elif metric_name == 'BIAS':
        return np.nanmean(sim - obs)
    elif metric_name == 'CORR':
        if np.std(obs) == 0 or np.std(sim) == 0:
            return np.nan
        return np.corrcoef(obs, sim)[0, 1]
    elif metric_name == 'KGE':
        r = np.corrcoef(obs, sim)[0, 1] if (np.std(obs) != 0 and np.std(sim) != 0) else np.nan
        if np.isnan(r) or np.mean(obs) == 0 or np.std(obs) == 0:
            return np.nan
        alpha = np.std(sim) / np.std(obs)
        beta = np.mean(sim) / np.mean(obs)
        return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    else:
        raise ValueError(f"Unknown metric: {metric_name}")

In [ ]:
RUN_PREPROCESSING = True # 是否重新进行数据预处理

In [ ]:
# 数据预处理，可以跳过直接读取缓存数据
if RUN_PREPROCESSING:
    # 数据预处理
    import xarray as xr
    from tqdm import tqdm
    ResultDict = {
        'Obs': {},
        CtrlName: {},
        ExpName: {},
                }
    # ---------------------------------------- 读取观测数据 ----------------------------------------
    with tqdm(total=YearNum, desc=f"Processing Obs") as pbar:
        for Year in Years:
            start_str, end_str =f'{Year}-06-01', f'{Year}-08-30'
            Read_StartDate, Read_EndDate = datetime.strptime(start_str, '%Y-%m-%d'), datetime.strptime(end_str, '%Y-%m-%d')
            ResultDict['Obs'][Year] = {}
            for Var in LSM_VarList:
                # 读取观测数据
                ObsFolder = obs_folder_mapping[Var]
                ObsVarName = obs_var_mapping[Var]
                ObsFileName = f"{ObsFolder}_{ObsVarName}_10km.{Year}.nc"
                ObsFilePath = os.path.join(Obs_DataPath, ObsFolder, 'dst', ObsFileName)
                ds = xr.open_dataset(ObsFilePath)
                ResultDict['Obs'][Year][Var] = ds[ObsVarName].sel(time=slice(Read_StartDate, Read_EndDate)).values * obs_var_ScaleFactor[Var]
            pbar.update(1)    
    pbar.close()
    # ---------------------------------------- 读取模拟数据 ----------------------------------------
    ExpStates = [CtrlName, ExpName]
    from tqdm import tqdm
    for ExpState in ExpStates:
        with tqdm(total=YearNum, desc=f"Processing {ExpState}") as pbar:
            for Year in Years:
                start_str, end_str = f'{Year}-06-01', f'{Year}-08-30'
                Read_StartDate, Read_EndDate = datetime.strptime(start_str, '%Y-%m-%d'), datetime.strptime(end_str, '%Y-%m-%d')
                ResultDict[ExpState][Year] = {}
                for Var in LSM_VarList: 
                    SimVarName = sim_var_mapping[Var]
                    SimFilePath = os.path.join(Sim_DataPath, f"{ExpState}_{SimVarName}_10km.{Year}.nc")
                    ds = xr.open_dataset(SimFilePath)
                    ResultDict[ExpState][Year][Var] = ds[SimVarName].sel(time=slice(Read_StartDate, Read_EndDate)).values * sim_var_ScaleFactor[Var]
                pbar.update(1)           
    pbar.close()   
    # ------------------------------- 处理为气候态数据（时间平均，计算空间评分）-----------------------------
    
    ClimateSpatialMeanResults = {}
    ClimateTemporalMeanResults = {}
    ClimateExtremeResults = {}
    ClimateResults= {}
    # ResultDict[ExpName][2024]['Prec'].shape
    for ExpState in ['Obs', CtrlName, ExpName]:
        ClimateSpatialMeanResults[ExpState] = {}
        ClimateTemporalMeanResults[ExpState] = {}
        ClimateResults[ExpState] = {}
        ClimateExtremeResults[ExpState] = {}
        for Var in LSM_VarList:
            ndays, ny, nx = ResultDict[ExpState][Year][Var].shape
            TemplateClimateVar = np.full((YearNum, ndays, ny, nx), np.nan)
            for YearIndex, Year in enumerate(Years):
                current_data = ResultDict[ExpState][Year][Var]
                actual_days = current_data.shape[0]
                TemplateClimateVar[YearIndex, :actual_days, :, :] = current_data
            ClimateResults[ExpState][Var] = TemplateClimateVar # (YearNum, ndays, ny, nx)
            ClimateSpatialMeanResults[ExpState][Var] = np.nanmean(np.nanmean(TemplateClimateVar, axis=0), axis=0) # (ny, nx)
            ClimateTemporalMeanResults[ExpState][Var] = np.nanmean(TemplateClimateVar, axis=0) # (ndays, ny, nx)

            if Var in ['T2max', 'Prec']:
                ClimateExtremeResults[ExpState][Var] = np.nanmax(ClimateResults[ExpState][Var], axis=1) # (YearNum, ny, nx)
            elif Var in ['T2min', 'SPEI']:
                ClimateExtremeResults[ExpState][Var] = np.nanmin(ClimateResults[ExpState][Var], axis=1) # (YearNum, ny, nx)

In [ ]:
def extract_SubsetRegion_mask(var ,domain_mask):
    row_indices, col_indices = np.where(domain_mask == 1)  # 提取 ==1 的区域
    row_start, row_end = row_indices.min(), row_indices.max()
    col_start, col_end = col_indices.min(), col_indices.max()
    # 判断数组维数
    if len(var.shape) == 2:
        # 步骤2：切片获取子区域数据（保持二维结构）
        sub_var = var[row_start:row_end+1, col_start:col_end+1]
    elif len(var.shape) == 3:
        # 步骤2：切片获取子区域数据（保持三维结构）
        sub_var = var[:, row_start:row_end+1, col_start:col_end+1]
    elif len(var.shape) == 4:
        # 步骤2：切片获取子区域数据（保持四维结构）
        sub_var = var[:, :, row_start:row_end+1, col_start:col_end+1]
    else:
        raise ValueError("输入数据的维度不正确，请输入二维或三维数组。")
    return sub_var
# =============================================================================
import cmaps
# 定义公共参数
Plt_CommonParams = {
    'width': 3.2,
    'Height': 2.3,
    'dpi': 250,
    # 'suptitle_size':7,
    'title_labelsize': 6.5,
    'xy_labelsize': 4.5,
    'cbar_Legend_size': 4.5,
    'ALPHA': 0.9,
    'Grid': True,
    'Save': False,
    'Shp': False,
    'River': False,
    'Basin': False,
    'Topo': False
}
# 定义变量参数
cbarTitle = True
cbar = True
subsample = 1 # 可视化提取数据间隔

quiverTitle = 'm/s'
CMAPmean=cmaps.MPL_StepSeq
# CMAPdiff=cmaps.amwg256
CMAPdiff = cmaps.NCV_manga
suptitle = None
ContourLabelsize = 5
DomainRegion = [21,41.5,107.7,122.6]  # 提取mask范围 [25, 35, 109, 122]
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'black'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.35     # 边框线宽
# 子区域
SubsetRegion = SubsetDomainConfigure[0]
DomainRegion2 = [21,41.5,108,123] # 提取mask范围 [25, 35, 109, 122]
DomainName = SubsetRegion["DomainName"]
DomainMask = create_domain_mask(lat_grid, lon_grid, DomainRegion2)
subLAT =extract_SubsetRegion_mask(lat_grid[ ::subsample, ::subsample] ,DomainMask[ ::subsample, ::subsample])
subLON =extract_SubsetRegion_mask(lon_grid[ ::subsample, ::subsample] ,DomainMask[ ::subsample, ::subsample])
# subDEM =extract_SubsetRegion_mask(DEM[ ::subsample, ::subsample] ,DomainMask[ ::subsample, ::subsample])
subLandMask = extract_SubsetRegion_mask(LandMask[ ::subsample, ::subsample] ,DomainMask[ ::subsample, ::subsample])

In [ ]:
# ---------------------------------- 子区域时间序列分析 ---------------------------------

import warnings
ExperimentTypes = ['Obs', CtrlName, ExpName]
SubRegionTemporalResults = {}
SubRegionSpatialResults = {}
SubRegionResults_3D = {}

for SubsetRegion in SubsetRegions:
    SubRegionTemporalResults[SubsetRegion] = {}
    SubRegionSpatialResults[SubsetRegion] = {}
    SubRegionResults_3D[SubsetRegion] = {}
    SubRegionDomainMask = create_domain_mask(lat_grid, lon_grid, SubsetRegions[SubsetRegion])

    SubRegionTemporalResults[SubsetRegion] = {}
    SubRegionSpatialResults[SubsetRegion] = {}
    SubRegionResults_3D[SubsetRegion] = {}
    for ExperimentType in ExperimentTypes:
        SubRegionTemporalResults[SubsetRegion][ExperimentType] = {}
        SubRegionSpatialResults[SubsetRegion][ExperimentType] = {}
        SubRegionResults_3D[SubsetRegion][ExperimentType] = {}
        for var in LSM_VarList:
            # SubRegionVar = extract_SubsetRegion_mask(ClimateResults[ExperimentType][var], SubRegionDomainMask) # 提取子区域变量 nyears, ndays, ny, nx
            SubRegionVar = extract_SubsetRegion_mask(ClimateExtremeResults[ExperimentType][var], SubRegionDomainMask)
            if var == 'SPEI':
                SubRegionVar = SubRegionVar * -1            
            SubRegionResults_3D[SubsetRegion][ExperimentType][var] = SubRegionVar # 计算子区域空间平均，得到 (time, lat, lon)
            # --- 空间平均场：对时间平均 (axis=0)，可能触发警告 ---
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                spatial_mean = np.nanmean(SubRegionVar, axis=0)
            SubRegionSpatialResults[SubsetRegion][ExperimentType][var] = spatial_mean # 计算子区域空间平均，得到 (lat, lon)


In [ ]:
def calculate_exceedance_probability(data, bins=100):
    hist, bin_edges = np.histogram(data, bins=bins, density=True)
    cdf = np.cumsum(hist) * np.diff(bin_edges)
    exceedance_prob = 1 - cdf
    return bin_edges[:-1], exceedance_prob
color_map = {
    'Obs': 'grey',
    'FSM': "#FF8C69",
    'Ctrl': "#73A1C2"
}
import matplotlib.font_manager as fm
# ========================= 计算分位数 =========================
percentiles = np.arange(0, 91, 10)                     # 百分位阈值（累积概率）
p_values = [p / 100.0 for p in percentiles]   # 转换为概率
UNIT = VarUnit[Var]
SubRegionExceedanceProbability = {}
for SubsetRegion in SubsetRegions:
    SubRegionExceedanceProbability[SubsetRegion] = {}
    for ExperimentType in ExperimentTypes:
        SubRegionExceedanceProbability[SubsetRegion][ExperimentType] = {}
        # SubRegionVar = ExtractSubsetData_byThreshold(config[Var]['X_ThresholdMax'], 
        #                                              config[Var]['X_ThresholdMin'], 
        #                                              SubRegionResults_3D[SubsetRegion][ExperimentType][Var], 
        #                                              IFravel=True) # 提取阈值范围内的变量
        SubRegionVar = SubRegionResults_3D[SubsetRegion][ExperimentType][Var]
        SubRegionVar = SubRegionVar[~np.isnan(SubRegionVar)] # 去除nan值
        # 计算超越概率分布
        bin_edges, exceedance_prob = calculate_exceedance_probability(SubRegionVar)
        SubRegionExceedanceProbability[SubsetRegion][ExperimentType][Var] = exceedance_prob
        SubRegionExceedanceProbability[SubsetRegion][ExperimentType][f"{Var}_bin_edges"] = bin_edges
SubRegionQuantiles = {}  # 存储各子区域各实验的分位数值
for SubsetRegion in SubsetRegions:
    SubRegionQuantiles[SubsetRegion] = {}
    for ExperimentType in ExperimentTypes:
        exceedance_prob = SubRegionExceedanceProbability[SubsetRegion][ExperimentType][Var]
        bin_edges = SubRegionExceedanceProbability[SubsetRegion][ExperimentType][f"{Var}_bin_edges"]
        # 计算累积分布函数 CDF = 1 - 超越概率
        cdf = 1 - exceedance_prob
        # 插值得到各百分位对应的变量值
        quantiles = {}
        for p in p_values:
            # 确保cdf单调递增（实际数据通常如此）
            # 若出现非单调，可用 np.interp 结合 np.unique 处理，此处简化为直接插值
            q = np.interp(p, cdf, bin_edges)
            quantiles[p] = q
        SubRegionQuantiles[SubsetRegion][ExperimentType] = quantiles

# ========================= 计算 BIAS =========================
BIAS_data = {}  # 存储各子区域各实验的偏差
for SubsetRegion in SubsetRegions:
    BIAS_data[SubsetRegion] = {}
    obs_quantiles = SubRegionQuantiles[SubsetRegion].get('Obs', None)
    if obs_quantiles is None:
        continue  # 若无观测数据则跳过
    for ExperimentType in ExperimentTypes:
        if ExperimentType == 'Obs':
            continue
        exp_quantiles = SubRegionQuantiles[SubsetRegion].get(ExperimentType, None)
        if exp_quantiles is None:
            continue
        bias = {p: exp_quantiles[p] - obs_quantiles[p] for p in p_values}
        BIAS_data[SubsetRegion][ExperimentType] = bias

# ========================= 可视化 =========================
fontsize = 8
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(6*0.7, 4*0.7), dpi=500, sharex=True) #, sharey=True
import matplotlib.ticker as ticker
for RegionIdx, SubsetRegion in enumerate(SubsetRegions):
    RowIdx, ColIdx = divmod(RegionIdx, 3)
    ax = axes[RowIdx, ColIdx]
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.6, alpha=0.7)  # 零参考线
    # 绘制各实验的偏差曲线
    for ExperimentType in ExperimentTypes:
        if ExperimentType == 'Obs':
            continue
        bias = BIAS_data[SubsetRegion].get(ExperimentType, None)
        if bias is None:
            continue
        x = percentiles                     # [20, 50, 90]
        y = [bias[p] for p in p_values]
        color = color_map.get(ExperimentType, 'black')
        if ExperimentType == 'Ctrl':
            ax.plot(x, y, label='CTRL', color=color, marker='o', linewidth=1.5, markersize=2, linestyle='--')
        else:
            ax.plot(x, y, label=ExperimentType, color=color, marker='o', linewidth=1.5, markersize=2)
    # 设置子图属性
    ax.set_title(f"({get_letter(RegionIdx+1)}) {SubsetRegion}", fontsize=fontsize+0.5, x=0.25, y=0.99)
    ax.set_xticks([20, 50, 90])
    ax.set_xticklabels([20, 50, 90])
    ax.tick_params(axis="both", which="major", direction="in", width=0.9, length=3, color='#363636', labelsize=fontsize)
    ax.tick_params(axis='x', labelsize=fontsize)
    ax.tick_params(axis='y', labelsize=fontsize-0.5)
    # 图例（仅在一个子图中显示）
    if RegionIdx == 4:
        fontprops = fm.FontProperties(size=fontsize+0.5)
        ax.legend(bbox_to_anchor=(2.3, 0.8), prop=fontprops, bbox_transform=ax.transAxes,
                  mode='compact', labelspacing=0.2, frameon=False)

    # 网格
    ax.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha=0.3)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(4))
# 删除多余子图
for i in range(RegionIdx+1, 6):
    fig.delaxes(axes[i//3, i%3])

# 统一坐标轴标签
fig.supxlabel('Percentile (%)', fontsize=fontsize+0.5, y=-0.01)
fig.supylabel(f'Bias ({UNIT})', fontsize=fontsize+0.5, x=0.02)

plt.subplots_adjust(wspace=0.3, hspace=0.2)
plt.show()
plt.close()

In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
def calculate_Mann_Whitney_U_significance_test_serial(X1, X2, IF_FDR = False, alpha=0.05):
    import numpy as np
    from scipy.stats import mannwhitneyu
    from statsmodels.stats.multitest import multipletests
    # 假设 X1: (n_strong, n_lat, n_lon)
    # X2:   (n_weak,   n_lat, n_lon)
    n_lat, n_lon = X1.shape[1], X1.shape[2]
    # 1. 计算中位数差（向量化）
    median_diff = np.nanmedian(X1, axis=0) - np.nanmedian(X2, axis=0)
    # 2. 处理 NaN：创建有效掩膜（所有样本在该格点非 NaN）
    valid_mask = (~np.isnan(X1).any(axis=0)) & (~np.isnan(X2).any(axis=0))
    # 3. 初始化结果数组
    p_values = np.full((n_lat, n_lon), np.nan)
    u_stats = np.full((n_lat, n_lon), np.nan)
    # 4. 提取有效格点的数据（只对 valid_mask 为 True 的位置计算）
    if valid_mask.any():
        # 将数据 reshape 为 (n_samples, n_valid_points)
        strong_flat = X1.reshape(X1.shape[0], -1)[:, valid_mask.ravel()]
        weak_flat = X2.reshape(X2.shape[0], -1)[:, valid_mask.ravel()]
        # 检查是否全为常数（可选，但 mannwhitneyu 在全等时会返回 p=1）
        # 实际上 mannwhitneyu 能处理这种情况，但会警告；我们可忽略或预处理
        # 使用 vectorized=True（默认在新版本中已启用）
        # 注意：alternative='two-sided' 是默认值（SciPy >= 1.7）
        u_stat_vec, p_val_vec = mannwhitneyu(
            strong_flat, weak_flat,
            axis=0,
            alternative='two-sided' # 双边检验
        )
        # 写回原形状
        p_values_flat = np.full(valid_mask.size, np.nan)
        u_stats_flat = np.full(valid_mask.size, np.nan)
        
        p_values_flat[valid_mask.ravel()] = p_val_vec
        u_stats_flat[valid_mask.ravel()] = u_stat_vec

        p_values = p_values_flat.reshape(n_lat, n_lon)
        u_stats = u_stats_flat.reshape(n_lat, n_lon)
    if IF_FDR:
        # 5. FDR 校正（向量化）
        p_flat = p_values.ravel()
        is_valid = ~np.isnan(p_flat)
        p_corrected = np.full_like(p_flat, np.nan)

        if is_valid.any():
            p_corrected[is_valid] = multipletests(p_flat[is_valid], method='fdr_bh')[1]
        p_values_fdr = p_corrected.reshape(p_values.shape)
        # 6. 显著性掩膜
        significant_fdr = (p_values_fdr < alpha) & (~np.isnan(p_values_fdr))
        return significant_fdr
    else:
        significant = (p_values < alpha) & (~np.isnan(p_values))
        return significant

def calculate_permutation_test_serial(X1, X2, var, n_perm=1000, alpha=0.1, IF_FDR=False, random_state=None):
    """
    Perform a two-sample permutation test on spatial data (e.g., strong vs weak monsoon years).
    
    Parameters:
    ----------
    X1 : array-like, shape (n_strong, n_lat, n_lon)
        Data for strong monsoon years.
    X2 : array-like, shape (n_weak,   n_lat, n_lon)
        Data for weak monsoon years.
    n_perm : int, default=5000
        Number of permutations.
    alpha : float, default=0.05
        Significance level.
    IF_FDR : bool, default=False
        Whether to apply FDR correction.
    random_state : int or None
        For reproducibility.

    Returns:
    -------
    significant : boolean array (n_lat, n_lon)
        True where difference is statistically significant.
    """
    rng = np.random.default_rng(random_state)
    
    X1 = np.asarray(X1)
    X2 = np.asarray(X2)
    n1, n_lat, n_lon = X1.shape
    n2 = X2.shape[0]
    n_total = n1 + n2

    # Original median difference
    orig_diff = np.nanmedian(X1, axis=0) - np.nanmedian(X2, axis=0)

    # Valid mask: all samples non-NaN at that grid point
    valid_mask = (~np.isnan(X1).any(axis=0)) & (~np.isnan(X2).any(axis=0))
    
    # Initialize p-values
    p_values = np.full((n_lat, n_lon), np.nan)

    if not valid_mask.any():
        # No valid points
        significant = np.zeros_like(p_values, dtype=bool)
        return significant

    # Flatten valid points for efficient computation
    valid_indices = np.where(valid_mask.ravel())[0]
    n_valid = len(valid_indices)

    # Stack data: (n_total, n_valid)
    X1_flat = X1.reshape(n1, -1)[:, valid_indices]  # (n1, n_valid)
    X2_flat = X2.reshape(n2, -1)[:, valid_indices]  # (n2, n_valid)
    X_combined = np.vstack([X1_flat, X2_flat])      # (n_total, n_valid)

    # Pre-allocate for permutation differences
    perm_diffs = np.empty((n_perm, n_valid))
    # 初始化进度条（仅当 n_perm >= 1000）
    pbar = tqdm(total=n_perm, desc=f"Permutations {var}") if n_perm >= 2000 else None
    for i in range(n_perm):
        shuffled_idx = rng.permutation(n_total)
        strong_idx = shuffled_idx[:n1]
        weak_idx = shuffled_idx[n1:]
        med_strong = np.median(X_combined[strong_idx], axis=0)
        med_weak = np.median(X_combined[weak_idx], axis=0)
        perm_diffs[i, :] = med_strong - med_weak
        if pbar:
            pbar.update(1)
    if pbar:
        pbar.close()
    # Observed absolute difference
    obs_abs = np.abs(orig_diff.ravel()[valid_indices])  # (n_valid,)

    # Count how many permuted |diff| >= observed |diff|
    # Use broadcasting: (n_perm, n_valid) vs (n_valid,)
    extreme_count = np.sum(np.abs(perm_diffs) >= obs_abs, axis=0)
    p_vals_valid = (extreme_count + 1) / (n_perm + 1)  # Add 1 for continuity correction

    # Fill back into full grid
    p_full = np.full(valid_mask.size, np.nan)
    p_full[valid_indices] = p_vals_valid
    p_values = p_full.reshape(n_lat, n_lon)

    # Apply FDR if requested
    if IF_FDR:
        p_flat = p_values.ravel()
        is_valid = ~np.isnan(p_flat)
        p_corrected = np.full_like(p_flat, np.nan)
        if is_valid.any():
            _, p_corr, _, _ = multipletests(p_flat[is_valid], alpha=alpha, method='fdr_bh')
            p_corrected[is_valid] = p_corr
        p_values = p_corrected.reshape(p_values.shape)

    # Determine significance
    significant = (p_values < alpha) & (~np.isnan(p_values))
    return significant
from concurrent.futures import ProcessPoolExecutor, as_completed
def process_single_variable(args):
    """
    并行处理单个变量的函数
    """
    var, X1, X2, n_perm, alpha, IF_FDR, random_state = args
    permutation_test_result = calculate_permutation_test_serial(
        X1, X2, var, 
        n_perm=n_perm, 
        alpha=alpha, 
        IF_FDR=IF_FDR, 
        random_state=random_state
    )
    mann_whitney_u_result = calculate_Mann_Whitney_U_significance_test_serial(X1, X2, IF_FDR = IF_FDR, alpha=alpha)
    return var, permutation_test_result, mann_whitney_u_result
def calculate_permutation_test_parallel(results, X1name, X2name, ReadVarList, n_perm=500, alpha=0.1, IF_FDR=True, random_state=42, n_processes=None):
    """
    使用ProcessPoolExecutor并行执行置换检验
    """
    
    # 准备参数列表
    task_args = []
    for var in ReadVarList:
        X1 = results[X1name][var][:]
        X2 = results[X2name][var][:]
        task_args.append((var, X1, X2, n_perm, alpha, IF_FDR, random_state))
    
    # 使用进程池并行执行
    SignificantResults = {
        'Mann_Whitney_U': {},
        # 'Permutation': {}
    }
    
    with ProcessPoolExecutor(max_workers=n_processes) as executor:
        # 提交所有任务
        future_to_var = {executor.submit(process_single_variable, args): args[0] for args in task_args}
        
        # 收集结果
        for future in tqdm(as_completed(future_to_var), total=len(task_args), desc="Processing variables"):
            var, permutation_test_result, mann_whitney_u_result = future.result()
            # SignificantResults['Permutation'][var] = permutation_test_result
            SignificantResults['Mann_Whitney_U'][var] = mann_whitney_u_result

    return SignificantResults


In [ ]:
# 计算统计显著性检验
ReadVarList = ['T2max', 'T2min', 'Prec']
X1name, X2name = 'Obs', 'Ctrl'
SignificantResultsBias = calculate_permutation_test_parallel(
    ClimateExtremeResults, 
    X1name,
    X2name,
    ReadVarList, 
    n_perm=500,  # 减少以加快示例运行
    alpha=0.1, 
    IF_FDR=True, 
    random_state=42,
    n_processes=15  # 限制进程数以便在示例中观察
)
X1name, X2name = 'FSM', 'Ctrl'
SignificantResultsDiff = calculate_permutation_test_parallel(
    ClimateExtremeResults, 
    X1name,
    X2name,
    ReadVarList, 
    n_perm=500,  # 减少以加快示例运行
    alpha=0.1, 
    IF_FDR=True, 
    random_state=42,
    n_processes=15  # 限制进程数以便在示例中观察
)

In [ ]:
# 
# 可视化
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.ticker as ticker
import cmaps
# 定义公共参数
Plt_CommonParams = {
    'width': 3.2,
    'Height': 2.3,
    'dpi': 250,
    'ALPHA': 0.85,
    'title_labelsize': 7,
    'xy_labelsize': 6,
    'cbar_Legend_size': 6,
    'Grid': True,
    'Save': False,
    'Shp': False,
    'River': False,
    'Basin': False,
    'Topo': True,
    'shrink': 0.8,
    'fraction': 0.05,
    # 'Cbar': False
}

def add_SubsetDomain(ax, SubsetDomain, linewidth, edgecolor, linestyle):
    import matplotlib.patches as patches
    rect = patches.Rectangle((SubsetDomain[2], SubsetDomain[0]),  # 左下角的坐标 (经度, 纬度)
                            SubsetDomain[3] - SubsetDomain[2],  # 宽度 (经度的范围)
                            SubsetDomain[1] - SubsetDomain[0],  # 高度 (纬度的范围)
                            linewidth=linewidth,  # 边框宽度
                            edgecolor=edgecolor,  # 边框颜色
                            facecolor='none',  # 矩形内部颜色，设置为 'none' 即可透明
                            linestyle=linestyle)  # 边框线型

    # 将矩形添加到坐标轴中
    ax.add_patch(rect) 
    pass
i = 0
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'grey'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.5     # 边框线宽
TotalVars = ['Obs', CtrlName, ExpName]
NumCols = len(TotalVars)
NumRows = 1
NumTicks = 5
pad = 0.1
subsample = 1



# 读取静态数据
with nc.Dataset('/home/yfdong/data/work/SMP/data/Domain/DEM_10km.nc') as ds:
    dstDEM = ds['DEM'][:]
with nc.Dataset('/home/yfdong/data/work/SMP/data/Domain/LandMask_10km.nc') as ds:
    landmask = ds['LandMask'][:]

plotMIN ,plotMAX1, PlotMax2, PlotMax3 = 0, 200, 80, 40
VarName = 'Prec'
CMAP = cmaps.WhiteBlueGreenYellowRed
num = 6 # 重采样

# plotMIN ,plotMAX1, PlotMax2, PlotMax3 = 305, 315, 5 , 4
# VarName = 'T2max'
# CMAP = cmaps.MPL_RdYlBu_r
# num = 10 # 重采样

plotMIN ,plotMAX1, PlotMax2, PlotMax3 = 280, 300, 5, 1
VarName = 'T2min'
CMAP = cmaps.MPL_RdYlBu_r
num = 10 # 重采样

landmask = np.where(landmask != 1, np.nan, landmask)

regrid_lon, regrid_lat = (lon_grid[::num, ::num], lat_grid[::num, ::num])
UNIT = VarUnit[VarName]
cbar = 'subcbar'
fig, axes = plt.subplots(nrows=NumRows, ncols=NumCols, figsize=(1.7*NumCols, 2.2*NumRows), dpi=700 ,sharex=True ,subplot_kw={'projection': ccrs.PlateCarree()})
# 绘制降水观测数据
# PltVar = np.nanmean(ClimateExtremeResults['Obs'][VarName], axis=0) *landmask
PltVar = np.nanmean(ClimateExtremeResults['Obs'][VarName], axis=0) *landmask
lon, lat = lon_grid, lat_grid
ax=axes[0]
DomainRegion = [21.6,41.5,107.7,123.1]
CMAP_Diff = cmaps.ncl_default
draw_2DVAR_subplot(
    DomainRegion, plotMIN ,plotMAX1, lon, lat, 
    PltVar[:] , f'(a) Obs ({UNIT})', cbarTitle=f'[{UNIT}]', 
    CMAP=CMAP, save_name=False,
    **Plt_CommonParams, NumTicks = NumTicks, pad=pad, DEM = dstDEM, ax=ax, 
    Cbar=cbar, IF_Rectangle=False
    )
# ================================ 绘制Ctrl与观测的偏差 ===============================
PltVar = np.nanmean((ClimateExtremeResults[CtrlName][VarName] - ClimateExtremeResults['Obs'][VarName]), axis=0) *landmask
TtestVar = SignificantResultsBias['Mann_Whitney_U'][VarName]*landmask
lon, lat = lon, lat
ax=axes[1]
draw_Bias_subplot(
    DomainRegion, PlotMax2, lon, lat, 
    PltVar[:] , f'(b) CTRL-Obs ({UNIT})', cbarTitle=f'[{UNIT}]', 
    CMAP=CMAP_Diff, save_name=False,
    **Plt_CommonParams,  NumTicks = NumTicks, pad=pad, DEM = dstDEM, ax=ax, 
    Cbar=cbar, IF_Rectangle=False
    )  
# 添加统计显著性检验

sig_indices = np.where(TtestVar[::num, ::num] >= 0.99)   # 获取显著格点索引
ax.scatter(regrid_lon[sig_indices], regrid_lat[sig_indices],
            marker='o', s=1, color='black', alpha=0.7, zorder=20)
# ============================ 绘制FSM与Ctrl的差异 ============================
TtestVar = SignificantResultsDiff['Mann_Whitney_U'][VarName] *landmask
PltVar = np.nanmean((ClimateExtremeResults['FSM'][VarName] - ClimateExtremeResults[CtrlName][VarName]), axis=0) *landmask
lon, lat = lon, lat
ax=axes[2]
draw_Bias_subplot(
    DomainRegion, PlotMax3, lon, lat, 
    PltVar[:] , f'(c) FSM-CTRL ({UNIT})', cbarTitle=f'[{UNIT}]', 
    CMAP=CMAP_Diff, save_name=False,
    **Plt_CommonParams,  NumTicks = NumTicks, pad=pad, DEM = dstDEM, ax=ax, 
    Cbar=cbar, IF_Rectangle=False
    ) 
# 添加统计显著性检验
sig_indices = np.where(TtestVar[::num, ::num] >= 0.99)   # 获取显著格点索引
ax.scatter(regrid_lon[sig_indices], regrid_lat[sig_indices],
            marker='o', s=1, color='black', alpha=0.7, zorder=20)
for axIdx, ax in enumerate(axes):
    ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.5, zorder=2)  # 绘制黑色海岸线
    # ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=2)  # 海洋填充为白色
    # ax.set_title(var, fontsize=Plt_CommonParams['xy_labelsize']+1 ) #,fontweight='bold'
    if axIdx != 0:
        ax.set_yticks([])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5))
    # ax.tick_params(axis='x', which='both', bottom=True, labelbottom=True)    
# 添加子区域
for ax in axes:
    for region in SubsetRegions:
        SubsetDomain = SubsetRegions[region] # 提取坐标范围
        add_SubsetDomain(ax, SubsetDomain, linewidth=1.5, edgecolor='black', linestyle='--')
# 调整子图间距
plt.subplots_adjust( wspace=0.05,hspace=0.05) #bottom=0.1, top=0.96,
plt.show()

In [ ]:
# ---------------------------------- 子区域年际变率序列分析 ---------------------------------
# SubsetRegions = {
#     # 'ECN': [21.5, 40, 108., 121], # Eastern China North
#     'HHP':[35, 40, 114., 119],
#     'LPT': [35, 40, 108, 114.],
#     'QDW': [28, 35, 108, 112.5], # Qinling-Daba-Wushan
#     'YHP': [28, 35, 112.5, 121], 
#     'SEH': [21.5, 28, 110, 120] # Southeastern Hills
# }
import warnings
ExperimentTypes = ['Obs', CtrlName, ExpName]
Yearly_SubRegionTemporalResults = {
    'Mean': {},
    'Anomaly': {},
}

for SubsetRegion in SubsetRegions:
    Yearly_SubRegionTemporalResults['Mean'][SubsetRegion] = {}
    SubRegionDomainMask = create_domain_mask(lat_grid, lon_grid, SubsetRegions[SubsetRegion])
    for ExperimentType in ExperimentTypes:
        Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType] = {}

        for var in LSM_VarList:
            SubRegionVar = extract_SubsetRegion_mask(ClimateExtremeResults[ExperimentType][var], SubRegionDomainMask) # 提取子区域变量
            Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var] = SubRegionVar # 计算子区域空间平均，得到 (year, lat, lon)
            Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var] = np.nanmean(SubRegionVar, axis=(1,2)) # 计算子区域时间序列，得到 (year,)

# 计算距平
for SubsetRegion in SubsetRegions:
    Yearly_SubRegionTemporalResults['Anomaly'][SubsetRegion] = {}
    for ExperimentType in ExperimentTypes:
        Yearly_SubRegionTemporalResults['Anomaly'][SubsetRegion][ExperimentType] = {}
        for var in LSM_VarList:
            Yearly_SubRegionTemporalResults['Anomaly'][SubsetRegion][ExperimentType][var] = Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var] - np.nanmean(Yearly_SubRegionTemporalResults['Mean'][SubsetRegion][ExperimentType][var], axis=0)


In [ ]:
# ================================== 子区域年际变率时间序列分析 ===================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
MeanVarRange = {
    # 'SSM': [0, 0.55],
    # 'ET': [1, 5],
    # 'Q2m': [5, 25],
    'T2max': [292, 305],

    # 'PSFC': [960, 990],
    # 'Wind10m': [0, 6],
    'Prec': [0, 15],
}
DiffVarRange = {
    # 'SSM': [-0.2, 0.2],
    # 'ET': [-0.6, 0.6],
    # 'Q2m': [-2, 2],    
    'T2max': [-4, 4],
    'T2min': [-3, 3],
    # 'PSFC': [-0.5, 5],
    # 'Wind10m': [-0.5, 5],
    'Prec': [-40, 40],
}
MetricName = 'CORR'
Type = 'Mean'  # 'Mean' or 'Anomaly'
Type = 'Anomaly'  # 'Mean' or 'Anomaly'
if Type == 'Mean':
    VarRange = MeanVarRange.copy()
elif Type == 'Anomaly':
    VarRange = DiffVarRange.copy()
# --------------------------------- 绘图模块 ------------------------------------
def PlotYearVar(Plot_result_dict , VarRange, VarName, subTitle,  ax1):
    # --------- 绘制土壤湿度模拟差异柱状图（左侧坐标轴）----------------------

    Read_StartDate, Read_EndDate = datetime(2000, 1, 1), datetime(2024, 12, 31)
    DateList = pd.date_range(start=Read_StartDate, end=Read_EndDate, freq='YE')
    ax1.set_ylim(VarRange[VarName][0], VarRange[VarName][1])

    ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax1.set_ylabel(f"{VarMap[VarName]} ({VarUnit[VarName]})", fontsize=fontsize, labelpad =10)
    LineWidth = 3

    # ----------- 绘制观测（右侧坐标轴）----------------------
    ax1.plot(DateList, Plot_result_dict['Obs'][VarName], 
            color= "#363636", marker='o',markersize= 3, markeredgewidth=0.1,
            linewidth = LineWidth , alpha=0.8, label="Obs", zorder=10) #,zorder=10  
    ax1.plot(DateList, Plot_result_dict[CtrlName][VarName], 
            color= "#73A1C2", 
            linewidth = LineWidth , alpha=1, label='CTRL', zorder=10, linestyle='--') #,zorder=10  
    ax1.plot(DateList, Plot_result_dict[ExpName][VarName], 
            color= "#FF8C69", 
            linewidth = LineWidth , alpha=1, label=ExpName, zorder=10) #,zorder=10  
    #如果是距平序列，则标出0线
    if Type == 'Anomaly':
        ax1.axhline(y=0, color='#FF0000', linestyle='--', linewidth=1.2, alpha=0.7)

    # 计算评分
    obs = Plot_result_dict['Obs'][VarName]
    CtrlSim = Plot_result_dict[CtrlName][VarName]
    ExpSim = Plot_result_dict[ExpName][VarName]
    CtrlMetric, ExpMetric = compute_metric(obs, CtrlSim, MetricName), compute_metric(obs, ExpSim, MetricName)
    ax1.text(0.5, 0.26, f"CTRL: {CtrlMetric:.2f}", # f"{MetricName}={CtrlMetric:.2f}"
             fontsize=fontsize-2., color="#009ACD", 
             bbox=dict(facecolor='none', edgecolor='none', boxstyle='round,pad=0.5', alpha=1), 
             transform=ax1.transAxes, va='top')
    ax1.text(0.5, 0.15, f"FSM: {ExpMetric:.2f}", # f"{MetricName}={CtrlMetric:.2f}"
             fontsize=fontsize-2., color="#FF4500", 
             bbox=dict(facecolor='none', edgecolor='none', boxstyle='round,pad=0.5', alpha=1), 
             transform=ax1.transAxes, va='top')
    # 时间标签设置
    ax1.xaxis.set_major_locator(mdates.YearLocator(base=10))
    # ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    # 旋转45度
    ax1.tick_params(axis='x', labelsize=fontsize-1, rotation=45)
    # 设置X轴范围
    ax1.set_xlim(datetime(2000, 1, 1), datetime(2024, 12, 31))
    # 设置刻度标签字体大小
    ax1.tick_params(axis='x', labelsize=fontsize-1.5)  # 设置x轴刻度标签大小
    ax1.tick_params(axis='y', labelsize=fontsize-1.5)  # 设置y轴刻度标签大小
    # 调整刻度线参数：
    ax1.minorticks_on()
    # 显示副刻度线
    ax1.tick_params(axis="both", which="major", direction="in", width=1, length=5, color='#363636')
    # grid 设置网格线性
    ax1.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha = 0.3)
    ax1.text(0.05, 0.85, subTitle, fontsize= fontsize,
            va='bottom', ha='left', transform=ax1.transAxes, weight='normal',  alpha=1)
    pass

fontsize = 15
Std_Thresold = 0.2
TemporalAnalysisLSM_VarList = ['T2max', 'T2min', 'Prec'] #'ET', 'Q2m', 
VarMap={
    'T2max': 'TXx',
    'T2min': 'TNn',
    'Prec': 'RX1day',
}
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'black'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.35     # 边框线宽
Cols = len(SubsetRegions)
Rows = len(TemporalAnalysisLSM_VarList)
fig, axes = plt.subplots(Rows, Cols, figsize=(Cols*2.8*0.9, Rows*2.8*0.9), dpi=400, 
                        constrained_layout=True, sharex=True)
for IDy, VarName in enumerate(TemporalAnalysisLSM_VarList):
    for IDx, SubsetRegion in enumerate(SubsetRegions):
        axeIndex = IDy*Cols + IDx
        subTitle = f"({get_letter(axeIndex)})"
        PlotYearVar(Yearly_SubRegionTemporalResults[Type][SubsetRegion], VarRange, VarName, subTitle, axes[IDy, IDx])
        if IDx != 0:
            axes[IDy, IDx].set_yticklabels([])  # 清除 y 轴刻度标签
            axes[IDy, IDx].set_ylabel('')       # 清除 y 轴标签
        if IDy == 0:
            axes[IDy, IDx].set_title(SubsetRegion, fontsize= fontsize, x= 0.5, y = 1.03, loc='center', weight='normal',  alpha=1)

# 创建统一的图例
lines, labels = fig.axes[-1].get_legend_handles_labels()
fig.legend( lines, labels,   
           fontsize= fontsize-0.5,        
            bbox_to_anchor=(0.66, 0.01),ncol=4, framealpha=1, edgecolor='none',
)
# 设置子图间距
plt.subplots_adjust(wspace=0.1, hspace=0.2)  # 调整宽度和高度间距
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed

# ============================================================
# 1. 指标函数：年际相关系数
# ============================================================
def compute_corr(obs, sim):
    """计算 obs 与 sim 的 Pearson 相关系数，自动质控 NaN"""
    obs = np.asarray(obs, dtype=np.float64)
    sim = np.asarray(sim, dtype=np.float64)
    valid = ~(np.isnan(obs) | np.isnan(sim))
    if valid.sum() < 3:
        return np.nan
    o, s = obs[valid], sim[valid]
    if np.std(o) == 0 or np.std(s) == 0:
        return np.nan
    return np.corrcoef(o, s)[0, 1]


def compute_kge(obs, sim):
    """计算 KGE，便于同时输出"""
    obs = np.asarray(obs, dtype=np.float64)
    sim = np.asarray(sim, dtype=np.float64)
    valid = ~(np.isnan(obs) | np.isnan(sim))
    if valid.sum() < 3:
        return np.nan
    o, s = obs[valid], sim[valid]
    if np.std(o) == 0 or np.mean(o) == 0:
        return np.nan
    r = np.corrcoef(o, s)[0, 1]
    alpha = np.std(s) / np.std(o)
    beta  = np.mean(s) / np.mean(o)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)


# ============================================================
# 2. 单个任务：对一组 (obs, ctrl, fsm) 做 year-block bootstrap
# ============================================================
def bootstrap_year_block(obs, ctrl, fsm, n_boot=500, seed=42,
                         metric_func=compute_corr):
    """
    以年为重采样单元，保持 Obs/Ctrl/FSM 的同年配对。
    返回 Δmetric 的 bootstrap 分布，以及 Ctrl/FSM 的 bootstrap 分布。
    """
    obs  = np.asarray(obs,  dtype=np.float64)
    ctrl = np.asarray(ctrl, dtype=np.float64)
    fsm  = np.asarray(fsm,  dtype=np.float64)

    n_year = len(obs)
    rng = np.random.default_rng(seed)

    delta_boot   = np.empty(n_boot)
    ctrl_boot    = np.empty(n_boot)
    fsm_boot     = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n_year, n_year)   # 有放回抽 25 年
        rc = metric_func(obs[idx], ctrl[idx])
        rf = metric_func(obs[idx], fsm[idx])
        ctrl_boot[b] = rc
        fsm_boot[b]  = rf
        delta_boot[b] = rf - rc

    return delta_boot, ctrl_boot, fsm_boot


# ============================================================
# 3. 汇总：从 bootstrap 分布计算 CI 和 p 值
# ============================================================
def summarize_bootstrap(delta_boot, ctrl_boot, fsm_boot, alpha=0.05):
    delta = delta_boot[~np.isnan(delta_boot)]
    if len(delta) < 10:
        return dict(Delta=np.nan, CI_low=np.nan, CI_high=np.nan,
                    p_value=np.nan, r_Ctrl=np.nan, r_FSM=np.nan)

    ci_low, ci_high = np.percentile(delta, [100*alpha/2, 100*(1-alpha/2)])
    # 双侧 p 值
    p = 2 * min(np.mean(delta <= 0), np.mean(delta >= 0))
    p = max(p, 1.0 / (len(delta) + 1))  # 防止 p=0

    return dict(
        Delta   = np.nanmean(delta),
        CI_low  = ci_low,
        CI_high = ci_high,
        p_value = p,
        r_Ctrl  = np.nanmean(ctrl_boot),
        r_FSM   = np.nanmean(fsm_boot),
    )


# ============================================================
# 4. 构建任务列表并并行执行
# ============================================================
LSM_VarList = ['T2max', 'T2min', 'Prec']
MetricFunc  = compute_corr   # 若同时想看 KGE，可换成 compute_kge
N_BOOT      = 500

tasks = []
for region in SubsetRegions:
    for var in LSM_VarList:
        obs  = Yearly_SubRegionTemporalResults['Anomaly'][region]['Obs'][var]
        ctrl = Yearly_SubRegionTemporalResults['Anomaly'][region][CtrlName][var]
        fsm  = Yearly_SubRegionTemporalResults['Anomaly'][region][ExpName][var]
        tasks.append((region, var, obs, ctrl, fsm))

# 并行执行；n_jobs=-1 使用全部核心；如内存不足可改成 n_jobs=4
boot_results = Parallel(n_jobs=-1, verbose=5)(
    delayed(bootstrap_year_block)(obs, ctrl, fsm,
                                  n_boot=N_BOOT,
                                  seed=42 + i,
                                  metric_func=MetricFunc)
    for i, (region, var, obs, ctrl, fsm) in enumerate(tasks)
)

# ============================================================
# 5. 汇总为 DataFrame，并做 FDR 多重比较校正
# ============================================================
rows = []
for (region, var, *_ ), (delta_boot, ctrl_boot, fsm_boot) in zip(tasks, boot_results):
    s = summarize_bootstrap(delta_boot, ctrl_boot, fsm_boot)
    s.update({'Region': region, 'Var': var})
    rows.append(s)

res_df = pd.DataFrame(rows)[
    ['Region', 'Var', 'r_Ctrl', 'r_FSM', 'Delta', 'CI_low', 'CI_high', 'p_value']
]

# FDR 校正（Benjamini-Hochberg，α=0.1）
valid_p = res_df['p_value'].notna()
res_df.loc[valid_p, 'p_adj'] = multipletests(
    res_df.loc[valid_p, 'p_value'].values,
    alpha=0.1, method='fdr_bh'
)[1]

# 显著性标记
res_df['Significant'] = np.where(
    (res_df['p_adj'] < 0.1) &
    ((res_df['CI_low'] > 0) | (res_df['CI_high'] < 0)),
    'Yes', 'No'
)

# 格式化输出
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(res_df.to_string(index=False))

# 保存结果
res_df.to_csv("extreme_Yearly_CORR_bootstrap_results.csv", index=False)